
#  Question 1 
> **Try a Support Vector Machine regressor (sklearn.svm.SVR), with various hyperparameters such as kernel="linear" (with various values for the C hyperparameter) or kernel="rbf" (with various values for the C and gammahyperparameters). Don’t worry about what these hyperparameters mean for now.How does the best SVR predictor perform?**


<table align="left">
  <td>
    <a target="_blank" href="https://www.kaggle.com/code/ossamasamir/chapter-3-from-book-hands-on-ml?scriptVersionId=363196832"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

In [ ]:
mnist = fetch_openml("mnist_784",version=1)

In [ ]:
mnist.keys()

In [ ]:
X = mnist["data"]
y = mnist["target"]

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
y = y.astype(np.uint8)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train , X_test, y_train, y_test = train_test_split(X,
                                                     y,
                                                     test_size=0.2,
                                                     random_state=42)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
kn = KNeighborsClassifier(n_neighbors=3)

In [ ]:
kn.fit(X_train,y_train)

In [ ]:
y_pred = kn.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
acc = accuracy_score(y_test,y_pred)

In [ ]:
acc

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
param_grid = {
    "n_neighbors": [3,4,5],
    "weights": ["uniform","distance"]
}

knn = KNeighborsClassifier()

grid_search = GridSearchCV(
    knn,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1
)

In [ ]:
X_train_2000 = X[:2000]
y_train_2000 = y[:2000]
X_test_1000 = X[2000:3000]
y_test_1000 = y[2000:3000]

In [ ]:
grid_search.fit(X_train_2000,y_train_2000)

In [ ]:
print(f"Best Paramete :{grid_search.best_params_}")
print(f"Best CV Accuracy :{grid_search.best_score_}")

In [ ]:
from sklearn.metrics import confusion_matrix

# Get predictions from the best model
y_presd_1000 = grid_search.best_estimator_.predict(X_test_1000)

# Calculate the final accuracy
acc = accuracy_score(y_test_1000,y_presd_1000)
# Generate the confusion matrix
cm = confusion_matrix(y_test_1000,y_presd_1000)
cm


In [ ]:
acc

In [ ]:

import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve
from sklearn.neighbors import KNeighborsClassifier


In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)

train_sizes, train_scores, val_scores = learning_curve(
    knn,
    X_train_2000, y_train_2000,
    train_sizes=[0.1, 0.3, 0.5, 0.7, 1.0],
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

In [ ]:
# Calculate the mean accuracy across CV folds
train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

In [ ]:
# Plot the curves
plt.plot(train_sizes, train_mean, marker="o", label="Training Accuracy")
plt.plot(train_sizes, val_mean, marker="s", label="Validation Accuracy")

plt.xlabel("Training Set Size")
plt.ylabel("Accuracy")
plt.title("Learning Curve - KNN on MNIST")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_curve, roc_auc_score

# Convert labels to binary: 5 vs. not 5
y_train_binary = (y_train_2000.astype(int) == 5).astype(int)
y_test_binary = (y_test_1000.astype(int) == 5).astype(int)

# Train a binary KNN model
knn_roc = KNeighborsClassifier(n_neighbors=3)
knn_roc.fit(X_train_2000, y_train_binary)

# Get probability of belonging to class 5
y_scores = knn_roc.predict_proba(X_test_1000)[:, 1]

# Calculate ROC and AUC
fpr, tpr, thresholds = roc_curve(y_test_binary, y_scores)
auc_score = roc_auc_score(y_test_binary, y_scores)

# Plot ROC Curve
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"KNN ROC (AUC = {auc_score:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Digit 5 vs. Other Digits")
plt.legend()
plt.grid(True)
plt.show()

print("AUC:", auc_score)
